# Featurizer Tutorial: Custom Primitives

This notebook demonstrates how to extend Featurizer with custom aggregation and transformation primitives using a financial analytics scenario.

> **Note:** Since the primitives expansion (commit `6b25339`), several primitives demonstrated here — Median, P95, Range, Log, ZScore — are now built-in. This tutorial is retained to show the **registration pattern** for creating truly custom primitives.

## 1. Built-in Primitives That Cover Common Needs

Before writing custom primitives, check if your need is already covered. Featurizer now ships with 43 aggregations and 71 transformations.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from featurizer.primitives.utils import list_aggregations, list_transformations

aggs = sorted(list_aggregations())
transforms = sorted(list_transformations())

print(f"Built-in aggregations ({len(aggs)}):")
print(f"  {', '.join(aggs)}")
print(f"\nBuilt-in transformations ({len(transforms)}):")
print(f"  {', '.join(transforms)}")

### Built-in Equivalents of This Example's Custom Primitives

| Custom (this example) | Built-in name | Type |
|---|---|---|
| `Median` | `median` | aggregation |
| `Percentile95` | `p95` | aggregation |
| `Range` | `range` | aggregation |
| `Log` | `ln` | transformation |
| `ZScore` | `cross_entity_zscore` | transformation |

To use built-in versions, no custom code is needed — just reference them by name.

## 2. The Custom Registration Pattern

When you need a truly custom primitive (e.g., domain-specific financial metrics), Featurizer provides a registration API. Let's walk through the pattern.

In [ ]:
with open("custom_primitives.py") as f:
    print(f.read())

### Key Steps:
1. **Subclass** `Aggregation` or `Transformation` from `featurizer.primitives.abstractions`
2. **Implement** `to_sql(feature, alias)` returning a SQL expression
3. **Register** via `register_aggregation()` or `register_transformation()`
4. **Call** `register_all_custom_primitives()` before creating the Featurizer

**Important**: Transformations must return new `Feature` instances (never mutate) to preserve hashing semantics.

## 3. Setup and Registration

In [ ]:
if not Path("data.db").exists():
    exec(open("create_data.py").read())

# Register custom primitives BEFORE creating Featurizer
from custom_primitives import register_all_custom_primitives

register_all_custom_primitives()

## 4. Create Featurizer with Custom Primitives

In [ ]:
from featurizer import Featurizer

featurizer = Featurizer("config.yaml")

print(f"Target entity: {featurizer.target.alias}")
print(f"Max depth: {featurizer.max_depth}")
print(f"Intervals: {featurizer.intervals}")

target_features = featurizer.features[featurizer.target.alias]
print(f"Generated features: {len(target_features)}")

## 5. Examining Features from Custom Primitives

Let's see which features use our custom primitives.

In [ ]:
custom_names = ["median", "p95", "range", "log", "zscore", "bin"]
custom_features = [
    f for f in target_features if any(prim in f.name.lower() for prim in custom_names)
]

print(f"Features using custom primitives: {len(custom_features)}")
for f in sorted(custom_features, key=lambda x: x.name)[:15]:
    print(f"  - {f.name}")

## 6. SQL Comparison: Custom vs Built-in

Let's compare the SQL generated by a custom primitive vs its built-in equivalent.

In [ ]:
sql = featurizer.query
print("Generated SQL (with custom primitives):")
print("=" * 80)
print(sql)
print("=" * 80)

## 7. Summary

In this tutorial, we learned:

1. **Check built-ins first**: Featurizer now ships with 43 aggregations and 71 transformations covering most common needs
2. **Registration pattern**: How to create custom primitives when built-ins aren't enough
3. **Implementation**: Subclass, implement `to_sql()`, register, use
4. **Best practices**: Always return new `Feature` instances from transformers

### When to Write Custom Primitives
- Domain-specific metrics (e.g., financial ratios, medical scores)
- Complex SQL expressions not covered by built-ins
- Organization-specific feature definitions

In [ ]:
print("Custom Primitives Summary")
print("=" * 40)
print(f"Built-in aggregations: {len(aggs)}")
print(f"Built-in transformations: {len(transforms)}")
print(f"Custom aggregations: 3 (median, p95, range)")
print(f"Custom transformations: 3 (log, zscore, bin)")
print(f"Total features generated: {len(target_features)}")